# ReplenishIQ - multi-agent replenishment copilot (demo)

A LangGraph **supervisor** coordinates four workers:

| Agent | What it does | LLM? |
|---|---|---|
| inventory_agent | stock + supplier terms via **MCP tools** (Streamable HTTP) | no |
| demand_agent | per-SKU forecast (pandas exponential smoothing) | no |
| policy_agent | **RAG** over policy docs, LLM extracts constraints, code checks grounding | yes (1 call) |
| optimizer_agent | **PuLP MILP** picks order quantities | no |

Then a **human approval** interrupt, the approved PO is written through MCP, and the LLM writes a short explanation.
About 3 Groq calls per run, which fits the free tier.

## 1. Setup
Clone the repo (on Colab) and install the requirements. Pip may warn about conflicts with Colab's preinstalled packages; you can ignore those warnings.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/rkoush/inventory-replenishment-multi-agent-copilot.git"
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    REPO = Path("/content/inventory-replenishment-multi-agent-copilot")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
else:  # running locally from notebooks/
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repo:", REPO)

In [ ]:
!pip install -q -r requirements.txt

## 2. Groq API key
Add `GROQ_API_KEY` in Colab's **Secrets** panel (key icon on the left) and turn on notebook access.
If there's no key, the notebook runs **offline**: rule-based stand-ins replace the 3 LLM steps, and everything else stays the same.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY") or ""
except Exception as e:  # not in Colab, or the secret is missing
    print("No Colab secret:", type(e).__name__)
os.environ.setdefault("GROQ_MODEL", "llama-3.3-70b-versatile")
# os.environ["GROQ_ROUTER_MODEL"] = "llama-3.1-8b-instant"   # optional: cheaper routing model
OFFLINE = not os.environ.get("GROQ_API_KEY")
print("Mode:", "OFFLINE (rule-based)" if OFFLINE else f"Groq {os.environ['GROQ_MODEL']}")

## 3. Synthetic retail data
This creates 10 SKUs at 2 stores with 180 days of sales (weekly seasonality and noise), supplier terms, and current stock. The data is seeded, so every run is the same.

In [ ]:
!python -m scripts.generate_data
import sqlite3, pandas as pd
con = sqlite3.connect("data/replenishiq.db")
pd.read_sql("SELECT * FROM inventory i JOIN suppliers s USING (sku) WHERE store='S001'", con)

## 4. RAG index over policy documents
Each markdown section becomes one chunk. Embeddings come from Chroma's local all-MiniLM-L6-v2 model, so no paid API is needed.

In [ ]:
!python -m rag.build_index
from rag.retriever import retrieve
for hit in retrieve("safety stock days by category", k=2):
    print(hit["score"], hit["source"], ">", hit["section"])

## 5. Start the MCP server (Streamable HTTP on :8000/mcp)
The server runs as a separate process. Agents find its tools at runtime through `langchain-mcp-adapters`.

In [ ]:
import subprocess, time, httpx
server = subprocess.Popen([sys.executable, "-m", "mcp_servers.inventory_server"],
                          stdout=open("mcp_server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if httpx.get("http://localhost:8000/health", timeout=1).status_code == 200:
            print("MCP server up:", httpx.get("http://localhost:8000/health").json()); break
    except httpx.HTTPError:
        time.sleep(0.5)
else:
    print(open("mcp_server.log").read())

In [ ]:
from agents.mcp_client import mcp_session
async with mcp_session() as tools:
    print("Tools:", tools.names)
    print(await tools.call("get_stock", sku="GRO-001", store="S001"))

## 6. Build the supervisor graph

In [ ]:
from agents.graph import build_graph
from agents.llm import get_llm
import config
llm = get_llm(offline=OFFLINE)
graph = build_graph(llm=llm, router_llm=get_llm(offline=OFFLINE, model=config.GROQ_ROUTER_MODEL))
print(graph.get_graph().draw_mermaid())

## 7. Run until the human-approval interrupt
Each line below is one agent step. The LangGraph checkpointer (MemorySaver) keeps the state while the graph is paused.

In [ ]:
from app.run_demo import stream, print_interrupt
cfg = {"configurable": {"thread_id": "colab-demo-1"}}
request = "Plan next week's replenishment for store S001 with a $5,000 budget"
payload = await stream(graph, {"request": request}, cfg)
print_interrupt(payload)

## 8. Human in the loop
Optional: send the plan back with a smaller budget. The optimizer runs again and the graph pauses again.

In [ ]:
from langgraph.types import Command
payload = await stream(graph, Command(resume={"decision": "revise", "budget": 4000, "approver": "rishi"}), cfg)
print_interrupt(payload)

Approve the plan. `place_order` then calls the MCP `create_purchase_order` tool, and the report node explains the plan.

In [ ]:
payload = await stream(graph, Command(resume={"decision": "approve", "approver": "rishi", "comment": "demo"}), cfg)
from IPython.display import Markdown, display
final = (await graph.aget_state(cfg)).values
display(Markdown(final["report"]))

In [ ]:
pd.read_sql("SELECT * FROM purchase_orders ORDER BY created_at DESC", con)

## 9. Tests (optional) and cleanup

In [ ]:
!python -m pytest -q -W ignore

In [ ]:
server.terminate(); print('MCP server stopped')